### 1. 패키지 설치

In [15]:
%pip install -qU langchain-openai langchain-pinecone langchain-unstructured "unstructured[docx]" python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2. 환경변수 로드

In [17]:
from dotenv import load_dotenv

load_dotenv()

True

### 3. 파싱 + 청킹

In [13]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    file_path='../../files/doc/tax.docx',
    chunking_strategy='by_title',
    max_characters=1000,
    overlap=200,
    languages=['kor', 'eng'],
    include_orig_elements=False
)

docs = loader.load()

### 4. 임베딩(설정)

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large'
)

### 5. 벡터 스토어
- 인덱스 준비

In [20]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()

index_name = 'tax-docx-index'

# 아래처럼 코드로 생성을 해도 되고, pinecone 대시보드에서 수동 생성해도 됨
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072, # text-embedding-3-large의 차원과 일치시킴
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

index = pc.Index(index_name)

- 벡터 스토어

In [23]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embedding
)

In [ ]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(docs))]

vector_store.add_documents(documents=docs, ids=uuids)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


['59f55e12-0b77-4fe9-8e89-3a96759b53ca',
 '80d7dc80-d063-41dd-99e4-ae62f1d688c8',
 'f87ad0bc-cfd1-43f4-a6b2-956b876f5cac',
 '3e92723d-0f15-4f5d-bcbc-9b00999ed473',
 'a0744240-5ae9-4d8e-a146-aa84e43d8d5f',
 '29635f53-7d09-4da8-b7b6-00ae87e86e05',
 '7390c0e6-7e2b-447d-be12-bf3f72bf1608',
 'ee1c7c1e-ee18-4c04-912d-99479552c03b',
 '515b10fa-4573-4034-9258-00a60d4f1f17',
 '144299d3-ac94-4e08-a246-1b595cccc4c7',
 '9d0592ea-738b-4d8e-a61a-f944ec60e5ae',
 '44224988-a4d7-4a80-a877-cd67f27e25e5',
 '091eecc2-41c0-417a-b6d0-3f384ce301b3',
 '265b4a62-ff8f-40ba-87e4-77b53de62fdd',
 '1abcef69-d610-4269-94ed-9470bd8a090c',
 '5466119a-91e5-4574-93a7-9bdcdd52c90e',
 'efc3ff41-d568-40c0-9620-2a09a73550e0',
 '7b37bea6-d614-444f-be2e-60894571d76f',
 'cbff916d-1080-4be7-a249-9eca65695077',
 'f80df298-348b-4318-844f-7a4cb9f508fb',
 '03e7b1ce-1aa0-4106-95b5-95902b7c34c3',
 'cd594c24-c44f-422e-a98b-f387f96b3526',
 'a9ae56df-5e91-44d6-87f2-b5b929b8df5e',
 '1408bb58-43c8-43f7-9d99-86b0bfd87be8',
 '82c25836-c8c3-

- 삭제

In [30]:
vector_store.delete(ids=[uuids[0], uuids[-1]])